# SDXL Studio — Google Colab T4 🎨⚡

สร้างภาพได้สูงสุด **1344 px** และเลือกสร้าง **1–2 ภาพต่อครั้ง** (ค่าที่ผู้ใช้ทดสอบแล้ว) พร้อมตัวช่วยเขียน prompt และ preset ตัวละครอนิเมะ 20 แบบ

## เริ่มใช้งาน
1. เลือก **Runtime → Change runtime type → T4 GPU**
2. รันเซลล์ติดตั้ง dependencies; ถ้ามีข้อความให้รีสตาร์ต runtime ให้รีสตาร์ต แล้วรันเซลล์ถัดไป
3. เลือกโมเดลและ Scheduler แล้วกด **Load model**
4. เลือก preset ตัวละคร (ไม่บังคับ) แล้วกด **ใส่ preset ลง Prompt** จากนั้นแก้ prompt ได้ตามต้องการ
5. เลือกขนาด/จำนวนภาพ แล้วกด **Generate**

> มีตัวเลือกขนาดตั้งแต่ 640 ถึง 1344 px; ขนาดใหญ่และ Batch 2 ใช้เวลาหรือ VRAM มากขึ้นตามโมเดลและ runtime หากพบ out of memory ให้ลด Batch หรือขนาดลง
> ลิงก์ Gradio ที่แชร์เป็นสาธารณะชั่วคราว อย่าใส่ข้อมูลส่วนตัวใน Prompt

In [ ]:
# ใช้ PyTorch ที่มากับ Colab เพื่อหลีกเลี่ยงการติดตั้ง CUDA/PyTorch ที่ไม่ตรงกับ runtime
!pip -q install -U diffusers transformers accelerate safetensors gradio

In [ ]:
import gc
import random
import torch
import gradio as gr
from diffusers import (
    StableDiffusionXLPipeline,
    EulerAncestralDiscreteScheduler,
    EulerDiscreteScheduler,
    UniPCMultistepScheduler,
    DDIMScheduler,
)

if not torch.cuda.is_available():
    raise RuntimeError("ไม่พบ GPU — เลือก Runtime → Change runtime type → T4 GPU แล้วรันใหม่")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name} | VRAM: {gpu_memory_gb:.1f} GB | dtype: float16")

MODELS = {
    "SSD-1B (เร็ว)": "segmind/SSD-1B",
    "Animagine XL 4.0": "cagliostrolab/animagine-xl-4.0",
    "Stable Diffusion XL Base 1.0": "stabilityai/stable-diffusion-xl-base-1.0",
}
SCHEDULERS = {
    "UniPC": UniPCMultistepScheduler,
    "Euler Ancestral": EulerAncestralDiscreteScheduler,
    "Euler": EulerDiscreteScheduler,
    "DDIM": DDIMScheduler,
}

# Preset เป็นจุดเริ่มต้นของ prompt; โมเดลอาจตีความตัวละครต่างกันตามรุ่น
CHARACTER_PRESETS = {
    "Hatsune Miku": "hatsune miku, aqua twintails, singing, stage outfit",
    "Asuka Langley": "asuka langley, red hair, blue eyes, plugsuit",
    "Rei Ayanami": "rei ayanami, short blue hair, red eyes, plugsuit",
    "Usagi Tsukino (Sailor Moon)": "usagi tsukino, long blonde twin tails, sailor senshi uniform",
    "Sakura Kinomoto": "sakura kinomoto, short brown hair, magical girl costume",
    "Rem": "rem (re:zero), blue hair, maid outfit, gentle smile",
    "Emilia": "emilia (re:zero), silver hair, purple eyes, white dress",
    "Zero Two": "zero two (darling in the franxx), pink hair, small red horns",
    "Nezuko Kamado": "nezuko kamado, long dark hair, bamboo muzzle, pink kimono",
    "Shinobu Kocho": "shinobu kocho, butterfly hair ornament, haori, purple eyes",
    "Marin Kitagawa": "marin kitagawa, blonde hair, gyaru fashion, cheerful",
    "Yor Forger": "yor forger, long black hair, red eyes, elegant black outfit",
    "Anya Forger": "anya forger, pink hair, green eyes, cute expression",
    "Frieren": "frieren, long white hair, elf ears, blue eyes, mage",
    "Fern": "fern (frieren), long purple hair, mage robe, calm expression",
    "Mikasa Ackerman": "mikasa ackerman, short black hair, red scarf, determined",
    "Power": "power (chainsaw man), blonde hair, red horns, sharp eyes",
    "Makima": "makima (chainsaw man), red hair, braided hair, golden eyes, suit",
    "Chisato Nishikigi": "chisato nishikigi, blonde hair, red ribbon, cheerful",
    "Rikka Takanashi": "rikka takanashi, eyepatch, black hair, chunibyo pose",
}

pipe = None

def load_model(model_label, scheduler_label):
    global pipe
    if pipe is not None:
        del pipe
        pipe = None
    gc.collect()
    torch.cuda.empty_cache()
    try:
        model_id = MODELS[model_label]
        scheduler = SCHEDULERS[scheduler_label].from_pretrained(model_id, subfolder="scheduler")
        new_pipe = StableDiffusionXLPipeline.from_pretrained(
            model_id, scheduler=scheduler, torch_dtype=torch.float16, use_safetensors=True
        ).to("cuda")
        new_pipe.enable_vae_slicing()
        new_pipe.enable_vae_tiling()
        attention_mode = "PyTorch SDPA"
        try:
            new_pipe.enable_xformers_memory_efficient_attention()
            attention_mode = "xFormers"
        except Exception:
            pass
        new_pipe.set_progress_bar_config(disable=True)
        pipe = new_pipe
        del new_pipe
        torch.cuda.empty_cache()
        return f"พร้อมใช้: {model_label} | {scheduler_label} | {attention_mode} | {gpu_name} ({gpu_memory_gb:.1f} GB)"
    except Exception as exc:
        pipe = None
        gc.collect()
        torch.cuda.empty_cache()
        raise gr.Error(f"โหลดโมเดลไม่สำเร็จ: {exc}")

def insert_character_preset(character, current_prompt):
    tags = CHARACTER_PRESETS.get(character, "")
    current_prompt = (current_prompt or "").strip()
    if not tags or tags in current_prompt:
        return current_prompt
    return f"{tags}, {current_prompt}" if current_prompt else tags

@torch.inference_mode()
def generate_images(prompt, negative_prompt, width, height, steps, cfg_scale, batch_size, seed):
    if pipe is None:
        raise gr.Error("กรุณากด Load model ก่อน")
    if not prompt or not prompt.strip():
        raise gr.Error("กรุณากรอก Prompt")
    seed = int(seed)
    if seed == -1:
        seed = random.randint(0, 2**31 - 1)
    elif seed < 0:
        raise gr.Error("Seed ต้องเป็น -1 (สุ่ม) หรือเลขตั้งแต่ 0 ขึ้นไป")
    try:
        generator = torch.Generator(device="cuda").manual_seed(seed)
        result = pipe(
            prompt=prompt.strip(), negative_prompt=(negative_prompt or "").strip() or None,
            width=int(width), height=int(height), num_inference_steps=int(steps),
            guidance_scale=float(cfg_scale), num_images_per_prompt=int(batch_size), generator=generator,
        )
        images = result.images
        del result
        return images, f"Seed: {seed}"
    except Exception as exc:
        gc.collect()
        torch.cuda.empty_cache()
        message = str(exc)
        if "out of memory" in message.lower():
            message += " — ลองลด Batch เป็น 1 หรือปรับขนาดภาพลง"
        raise gr.Error(f"สร้างภาพไม่สำเร็จ: {message}")

In [ ]:
sizes = [640, 768, 832, 896, 1024, 1216, 1344]

with gr.Blocks(theme=gr.themes.Soft(), title="SDXL Studio for T4") as demo:
    gr.Markdown("# SDXL Studio ⚡\nรองรับภาพสูงสุด 1344 px และ Batch 2 — ใช้ค่าที่ทดสอบแล้ว")

    with gr.Accordion("1. โหลดโมเดล", open=True):
        with gr.Row():
            model_choice = gr.Dropdown(choices=list(MODELS), value="SSD-1B (เร็ว)", label="โมเดล")
            scheduler_choice = gr.Dropdown(choices=list(SCHEDULERS), value="UniPC", label="Scheduler")
        load_button = gr.Button("Load model", variant="secondary")
        status = gr.Textbox(label="สถานะ GPU / โมเดล", value="ยังไม่ได้โหลดโมเดล", interactive=False)
        load_button.click(load_model, inputs=[model_choice, scheduler_choice], outputs=status)

    with gr.Accordion("แนวทางเขียน Prompt + ตัวละคร 20 แบบ", open=True):
        gr.Markdown("**โครงสร้างง่าย ๆ:** ตัวละคร/สิ่งหลัก + ลักษณะหรืออารมณ์ + กำลังทำอะไร + ฉาก/สถานที่ + แสง/มุมกล้อง + สไตล์\n\nตัวอย่าง: `a young mage reading a book, cozy library, warm window light, full body, detailed anime illustration`\n\nเขียนเป็นคำหรือวลีที่ชัดเจน เริ่มจากสิ่งสำคัญก่อน แล้วค่อยเติมรายละเอียดทีละส่วน; Negative prompt ใช้ระบุสิ่งที่ไม่อยากได้ เช่น `blurry, low quality, extra fingers, watermark`")
        with gr.Row():
            character_choice = gr.Dropdown(choices=["ไม่ใช้ preset"] + list(CHARACTER_PRESETS), value="ไม่ใช้ preset", label="เลือก preset ตัวละคร")
            apply_character_button = gr.Button("ใส่ preset ลง Prompt")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 2. เขียน Prompt และตั้งค่าภาพ")
            prompt = gr.Textbox(label="Prompt", lines=5, placeholder="ตัวละคร/สิ่งหลัก, ลักษณะ, การกระทำ, ฉาก, แสง, มุมกล้อง, สไตล์")
            negative_prompt = gr.Textbox(label="Negative prompt (ไม่บังคับ)", lines=2, placeholder="เช่น blurry, low quality, extra fingers, watermark")
            with gr.Row():
                width = gr.Dropdown(choices=sizes, value=1024, label="กว้าง (px)")
                height = gr.Dropdown(choices=sizes, value=1024, label="สูง (px)")
            with gr.Row():
                steps = gr.Slider(minimum=10, maximum=40, step=1, value=20, label="Steps")
                cfg_scale = gr.Slider(minimum=1, maximum=10, step=0.5, value=5, label="CFG Scale")
            with gr.Row():
                batch_size = gr.Slider(minimum=1, maximum=2, step=1, value=2, label="จำนวนภาพ (Batch 1–2)")
                seed = gr.Number(value=-1, precision=0, label="Seed (-1 = สุ่ม)")
            generate_button = gr.Button("Generate", variant="primary")
        with gr.Column(scale=1):
            gr.Markdown("### 3. ผลลัพธ์")
            gallery = gr.Gallery(label="ภาพที่สร้าง", columns=2, height="auto")
            seed_info = gr.Textbox(label="Seed ที่ใช้", interactive=False)

    apply_character_button.click(
        fn=insert_character_preset, inputs=[character_choice, prompt], outputs=prompt
    )
    generate_button.click(
        fn=generate_images,
        inputs=[prompt, negative_prompt, width, height, steps, cfg_scale, batch_size, seed],
        outputs=[gallery, seed_info],
    )

# เข้าคิวทีละงานเพื่อป้องกันการใช้ VRAM ซ้อนกัน
demo.queue().launch(share=True, inline=True)